In [ ]:
import pandas as pd
import pickle
import arviz as az
from plotly import graph_objects as go
from datetime import datetime
import pycountry

from emu_renewal.inputs import DATA_PATH
from emu_renewal.outputs import get_output_dir, load_targets
from emu_renewal.plotting import plot_spaghetti_calib_comparison, plot_post_prior_comparison, plot_imm_props

In [ ]:
country = "France"
analysis_times = {
    "no_mob": "20250121_2159",
    "google_nonresi_linear": "20250121_2116",
    "google_nonresi_square": "20250121_2120",
    "fb_linear": "20250121_2133",
    "fb_square": "20250121_2146",
}

In [ ]:
spaghs = {}
targets = {}
idatas = {}
mobs = {}
for analysis, time in analysis_times.items():
    outdir = get_output_dir(country, analysis, time)
    spaghs[analysis] = pd.read_hdf(outdir / "spaghetti.h5")
    targets[analysis] = load_targets(country, analysis, time)
    idatas[analysis] = az.from_netcdf(outdir / "idata.nc")
    mobs[analysis] = pd.read_csv(DATA_PATH / f"mobility/{pycountry.countries.get(name=country).alpha_2}_mob_data.csv", index_col=0)

In [ ]:
def plot_spaghetti_calib_comparison(spaghetti, calib_data, out_req):
    fig = make_subplots(
        rows=len(out_req),
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.02,
        horizontal_spacing=0.05,
        subplot_titles=out_req,
    ).update_layout(height=300*len(out_req), width=800, showlegend=False)
    out_style = {"color": "black", "width": 0.5}
    targ_style = {"color": "red"}
    for o, out in enumerate(out_req):
        for col in spaghetti[out].columns:
            line = go.Scatter(x=spaghetti.index, y=spaghetti[out][col], line=out_style)
            fig.add_trace(line, row=o+1, col=1)
        if out in calib_data:
            target = calib_data[out]
            target_scatter = go.Scatter(x=target.index, y=target, mode="markers", line=targ_style)
            fig.add_trace(target_scatter, row=o+1, col=1)
    return fig

In [ ]:
outputs = list(targets.keys()) + ["process", "seropos"]
spagh_plot = plot_spaghetti_calib_comparison(spaghetti, targets, outputs)
spagh_plot.add_traces(go.Scatter(x=filtered_mob.index, y=filtered_mob, line={"color": "blue"}), rows=outputs.index("process") + 1, cols=1)

In [ ]:
plot_imm_props(spaghetti, 2)

In [ ]:
priors = pickle.load(open(outdir / "priors.pkl", "rb"))
epi_params = [p for p in priors if p != "shared_dispersion"]
plot_post_prior_comparison(idata, epi_params, priors, req_grid=[8, 2], req_size=[10, 40]);